# 07. Сборка RuREBus corrected_v1

Создаёт неизменяемую версию датасета из `rurebus_data/processed` и применяет только решения `ACCEPTED`. Сборка атомарная; существующая версия никогда не перезаписывается.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import runpy
PROJECT_DIR = Path('/content/drive/MyDrive/NER_RuREBus_project')
runpy.run_path(str(PROJECT_DIR / 'colab_bootstrap.py'))['bootstrap_project'](PROJECT_DIR)
SOURCE = PROJECT_DIR / 'rurebus_data/processed'
OUTPUT = PROJECT_DIR / 'rurebus_data/versions/corrected_v1'
ARCHIVE = PROJECT_DIR / 'rurebus_data/versions/corrected_v1.zip'
CORRECTIONS = PROJECT_DIR / 'configs/data/corrections/rurebus_corrected_v1.csv'
INTEGRITY = PROJECT_DIR / 'results/data_audit/rurebus_conflict_resolution_v1/file_integrity.csv'

In [ ]:
from collections import Counter
from rurebus_ie.data.conflict_resolution import read_corrections
from rurebus_ie.data.dataset_versioning import build_versioned_dataset, validate_versioned_dataset
corrections = read_corrections(CORRECTIONS)
print('Решения:', Counter(row.decision_status for row in corrections))
if OUTPUT.exists():
    print('Версия уже существует; выполняется только валидация.')
    report = validate_versioned_dataset(OUTPUT)
elif ARCHIVE.is_file():
    from tempfile import TemporaryDirectory
    from zipfile import ZipFile
    OUTPUT.parent.mkdir(parents=True, exist_ok=True)
    with TemporaryDirectory(prefix='.corrected_v1-install-', dir=OUTPUT.parent) as temporary:
        staged = Path(temporary) / OUTPUT.name
        staged.mkdir()
        with ZipFile(ARCHIVE) as archive:
            staged_root = staged.resolve()
            for member in archive.infolist():
                target = (staged / member.filename).resolve()
                if target != staged_root and staged_root not in target.parents:
                    raise ValueError(f'Небезопасный путь в ZIP: {member.filename}')
            archive.extractall(staged)
        report = validate_versioned_dataset(staged)
        staged.replace(OUTPUT)
    print('Архив установлен и проверен:', ARCHIVE)
else:
    report = build_versioned_dataset(
        SOURCE, OUTPUT, corrections,
        correction_manifest_path=CORRECTIONS,
        dataset_version='corrected_v1',
        parent_version='original_processed_v1',
        source_integrity_manifest=INTEGRITY,
        allowed_statuses=('ACCEPTED',),
    )
report

In [ ]:
import pandas as pd
manifest = pd.read_csv(OUTPUT / 'manifest.csv', encoding='utf-8-sig')
display(manifest.groupby('split').agg(documents=('document_id','count'), entities=('entity_count','sum'), relations=('relation_count','sum')))
print('Manifest:', OUTPUT / 'manifest.csv')
print('Report:', OUTPUT / 'dataset_report.json')